# Relay — Generic Self-Service Hugging Face to vLLM on Kaggle

Deploy **any supported Hugging Face model** (causal text LMs or multimodal vision/OCR LMs) via **vLLM 0.29.0** on Kaggle (2× NVIDIA Tesla T4 GPUs) and expose it securely to **Relay** using an encrypted **Cloudflare Quick Tunnel**.

### System Architecture
```text
Local Relay Gateway  ──►  Cloudflare Quick Tunnel  ──►  vLLM Engine (Auto-Recommended TP)  ──►  Hugging Face Model
 (developer machine)     (*.trycloudflare.com)       (port 8000 on dual T4)     (e.g., Qwen2.5, GLM-OCR, Llama-3)
```

### How to Use this Notebook
1. **Edit the first configuration cell below** with your desired `MODEL_ID` (supports causal text models and multimodal models like `zai-org/GLM-OCR`).
2. Click **Run All** (or `Shift + Enter` through the cells).
3. Everything else—hardware checks, preflight inspection, vLLM startup, readiness polling, local smoke tests (text or OCR), and Cloudflare tunnel provisioning—is fully automated.
4. Copy the resulting Relay configuration snippet from the final output cells into your local `.env`.


---
## Configuration (The ONLY cell you need to edit)
Specify your target Hugging Face model ID. By default, all deployment parameters are set to `None` to enable the **automatic memory-aware recommendation engine**. Preflight will inspect model metadata, calculate attention dimensions & memory footprint, evaluate GPU capacity, and recommend optimal parameters.

You may optionally specify explicit overrides for any parameter (`TENSOR_PARALLEL_SIZE`, `MAX_MODEL_LEN`, `GPU_MEMORY_UTILIZATION`, etc.) if needed.


In [ ]:
import os

# ==============================================================================
# Relay — Automatic Recommendation & Deployment Configuration
# ==============================================================================

# 1. Target Hugging Face Model
# Supported model architectures:
#   - Text Causal LMs: Qwen/Qwen2.5-Coder-7B-Instruct, meta-llama/Llama-3.1-8B-Instruct, mistralai/Mistral-7B-Instruct-v0.3
#   - Multimodal Vision/OCR LMs: zai-org/GLM-OCR
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

# To deploy GLM-OCR instead, simply uncomment:
# MODEL_ID = "zai-org/GLM-OCR"

# 2. Deployment Overrides (Default: None for Automatic Recommendation)
# When set to None, preflight inspects model facts, calculates memory footprint,
# and automatically recommends optimal serving settings.
SERVED_MODEL_NAME = None      # Auto-resolved: sanitized model alias (e.g., "qwen2.5-coder-7b-instruct")
TENSOR_PARALLEL_SIZE = None   # Auto-recommended: 1 for smaller models, 2 when workload requires dual GPUs
MAX_MODEL_LEN = None          # Auto-recommended: bounded to native context (<= 4096)
MAX_NUM_SEQS = None           # Auto-recommended: 1 (conservative smoke-test default)
GPU_MEMORY_UTILIZATION = None # Auto-calculated from estimated memory requirements [0.70 - 0.92]
DTYPE = None                  # Auto-resolved: "float16" on Tesla T4 (Turing CC 7.5)
QUANTIZATION = None           # Auto-detected from HF config (e.g. "awq", "gptq")
TRUST_REMOTE_CODE = None      # Auto-resolved: True if auto_map is defined, else False
EXTRA_VLLM_ARGS = None        # Optional extra CLI flags (e.g. "--limit-mm-per-prompt image=1")

# 3. Hugging Face Access Token (Required for gated/private models like Llama 3)
# Recommended: Add 'HF_TOKEN' in Kaggle -> Add-ons -> Secrets
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = HF_TOKEN or UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

# Export parameters to os.environ so all shell scripts and preflight can read them
os.environ["MODEL_ID"] = MODEL_ID

def _sync_env(key, val):
    if val is not None:
        os.environ[key] = str(val)
    else:
        os.environ.pop(key, None)

_sync_env("SERVED_MODEL_NAME", SERVED_MODEL_NAME)
_sync_env("TENSOR_PARALLEL_SIZE", TENSOR_PARALLEL_SIZE)
_sync_env("MAX_MODEL_LEN", MAX_MODEL_LEN)
_sync_env("MAX_NUM_SEQS", MAX_NUM_SEQS)
_sync_env("GPU_MEMORY_UTILIZATION", GPU_MEMORY_UTILIZATION)
_sync_env("DTYPE", DTYPE)
_sync_env("QUANTIZATION", QUANTIZATION)
if TRUST_REMOTE_CODE is not None:
    os.environ["TRUST_REMOTE_CODE"] = str(TRUST_REMOTE_CODE).lower()
else:
    os.environ.pop("TRUST_REMOTE_CODE", None)
_sync_env("EXTRA_VLLM_ARGS", EXTRA_VLLM_ARGS)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = str(HF_TOKEN)

token_indicator = 'Yes (Masked)' if HF_TOKEN else 'No (Public models only)'
print(f"Model ID:          {MODEL_ID}")
print(f"Served Alias:      {SERVED_MODEL_NAME or '(Auto-recommended)'}")
print(f"Tensor Parallel:   {TENSOR_PARALLEL_SIZE if TENSOR_PARALLEL_SIZE is not None else '(Auto-recommended)'}")
print(f"Max Model Len:     {MAX_MODEL_LEN if MAX_MODEL_LEN is not None else '(Auto-recommended)'}")
print(f"Max Num Seqs:      {MAX_NUM_SEQS if MAX_NUM_SEQS is not None else '(Auto-recommended: 1)'}")
print(f"GPU Memory Util:   {GPU_MEMORY_UTILIZATION if GPU_MEMORY_UTILIZATION is not None else '(Auto-calculated)'}")
print(f"HF Token Provided: {token_indicator}")


---
## Step 1: GPU & Hardware Verification
Confirm two NVIDIA Tesla T4 GPUs are allocated with available VRAM.


In [ ]:
import subprocess
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("nvidia-smi not found. Select 'GPU T4 x2' in Kaggle Notebook Settings.")

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv"], check=False)

try:
    import torch
    dev_count = torch.cuda.device_count()
    print(f"\nPyTorch CUDA Devices: {dev_count}")
    for i in range(dev_count):
        free_mb = torch.cuda.mem_get_info(i)[0] / (1024 ** 2)
        total_mb = torch.cuda.mem_get_info(i)[1] / (1024 ** 2)
        print(f"  GPU {i} ({torch.cuda.get_device_name(i)}): {free_mb:.0f} MiB free / {total_mb:.0f} MiB total")
    
    if TENSOR_PARALLEL_SIZE is not None and dev_count < TENSOR_PARALLEL_SIZE:
        print(f"\n[WARNING] Detected {dev_count} GPU(s), but TENSOR_PARALLEL_SIZE is {TENSOR_PARALLEL_SIZE}.")
    else:
        print("\n[PASS] Hardware verified.")
except Exception as e:
    print(f"PyTorch CUDA check error: {e}")


---
## Step 2: Clone or Update Relay Repository
Ensures management scripts in `infra/kaggle/` are present and up to date.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/mosabbir-maruf/Relay.git"
CLONE_DEST = "/kaggle/working/Relay"

if not os.path.exists(CLONE_DEST):
    print(f"Cloning {REPO_URL} -> {CLONE_DEST}...")
    subprocess.run(["git", "clone", REPO_URL, CLONE_DEST], check=True)
else:
    print(f"Updating repository at {CLONE_DEST}...")
    subprocess.run(["git", "-C", CLONE_DEST, "pull", "--ff-only"], check=False)

os.chdir(CLONE_DEST)
print(f"Working directory: {os.getcwd()}")


---
## Step 3: Apply Script Permissions (`chmod +x`)
Applies POSIX execute permissions to shell scripts in `infra/kaggle/`.


In [ ]:
import glob
import os
import subprocess

for script in sorted(glob.glob("infra/kaggle/*.sh")):
    subprocess.run(["chmod", "+x", script], check=True)
    print(f"chmod +x {script} -> OK")


---
## Step 4: Run Preflight Compatibility & Automatic Recommendation
Queries Hugging Face Hub metadata, extracts architecture facts & attention dimensions, calculates memory footprint (weights, visual encoder, KV cache, CUDA runtime), evaluates single vs dual GPU fit, checks vLLM version requirements, and generates candidate recommendations.


In [ ]:
import subprocess

# Run preflight inspection using configured environment variables
subprocess.run(["./infra/kaggle/vllm.sh", "preflight"], check=True)


---
## Step 5: Check & Verify Packages (vLLM and Transformers)
Verifies that `vllm==0.29.0` is installed, enforces `transformers>=5.3.0` for multimodal architectures (such as `GLM-OCR`), and prints runtime package versions.


In [ ]:
import subprocess
import shutil

# 1. Verify vLLM CLI installation
if not shutil.which("vllm"):
    print("vLLM CLI not found. Installing verified vLLM 0.29.0 (~60s)...")
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "vllm==0.29.0"], check=True)

# 2. Enforce transformers >= 5.3.0 for multimodal models (e.g. GLM-OCR) without touching pinned vLLM
try:
    import transformers
    from packaging import version
    need_tf_install = version.parse(transformers.__version__) < version.parse("5.3.0")
except Exception:
    need_tf_install = True

if need_tf_install:
    print("Installing/upgrading transformers >= 5.3.0 for multimodal architecture support...")
    subprocess.run(["pip", "install", "-q", "transformers>=5.3.0"], check=True)

# 3. Print verified runtime versions
import transformers
print(f"Transformers version: {transformers.__version__}")
subprocess.run(["vllm", "--version"], check=True)

# 4. Run environment check
subprocess.run(["./infra/kaggle/vllm.sh", "check"], check=True)


---
## Step 6: Safe Process & Port 8000 Cleanup
Terminates stale listeners on port 8000 and releases leaked GPU VRAM.


In [ ]:
import subprocess

subprocess.run(["./infra/kaggle/vllm.sh", "clean"], check=True)


---
## Step 7: Launch vLLM in Background
Executes the resolved `vllm serve` command in the background with nohup. Logs stream to `/kaggle/working/vllm_server.log`.


In [ ]:
import subprocess

subprocess.run(["./infra/kaggle/vllm.sh", "start"], check=True)


---
## Step 8: Wait for vLLM Readiness
Polls `http://127.0.0.1:8000/v1/models` until weights are downloaded and the KV cache compiles.


In [ ]:
import time
import urllib.request
import json
import subprocess

HEALTH_URL = "http://127.0.0.1:8000/v1/models"
TIMEOUT_SECS = 600
POLL_INTERVAL = 5

print(f"Waiting for vLLM readiness at {HEALTH_URL} (timeout: {TIMEOUT_SECS}s)...")
start_time = time.time()
ready = False

while time.time() - start_time < TIMEOUT_SECS:
    elapsed = int(time.time() - start_time)
    try:
        req = urllib.request.Request(HEALTH_URL)
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode("utf-8"))
                models = [m.get("id") for m in data.get("data", [])]
                print(f"\n[READY] vLLM is online after {elapsed}s!")
                print(f"Available models: {models}")
                ready = True
                break
    except Exception:
        if elapsed % 20 == 0:
            print(f"[WAITING] Loading weights and compiling KV cache... ({elapsed}s elapsed)")
    time.sleep(POLL_INTERVAL)

if not ready:
    print("\n[TIMEOUT] vLLM did not become ready in time. Last 30 log lines:")
    subprocess.run(["./infra/kaggle/vllm.sh", "logs", "30"], check=False)
    raise TimeoutError("vLLM readiness timeout. Inspect log output above.")


---
## Step 9: Local Model Check
Verifies that `/v1/models` exposes the configured served model name alias.


In [ ]:
import urllib.request
import json

req = urllib.request.Request("http://127.0.0.1:8000/v1/models")
with urllib.request.urlopen(req, timeout=5) as resp:
    data = json.loads(resp.read().decode("utf-8"))

models = [m["id"] for m in data.get("data", [])]
print(f"Models returned by /v1/models: {models}")
assert SERVED_MODEL_NAME in models, f"Expected '{SERVED_MODEL_NAME}' to be registered, got {models}"
print(f"[PASS] Model alias '{SERVED_MODEL_NAME}' confirmed online.")


---
## Step 10: Local Inference Smoke Test
Sends a test prompt completion to the local vLLM endpoint.


In [ ]:
import subprocess

subprocess.run(["./infra/kaggle/vllm.sh", "test"], check=True)


---
## Step 11: Local Streaming Test (SSE)
Verifies real-time Server-Sent Events (SSE) token streaming and validates terminal `data: [DONE]` frame.


In [ ]:
import urllib.request
import json
import sys

# Detect capability (text vs multimodal)
is_multimodal = False
try:
    sys.path.insert(0, "infra/kaggle")
    import preflight
    info, cfg = preflight.fetch_hf_model_metadata(MODEL_ID, os.environ.get("HF_TOKEN"))
    attrs = preflight.inspect_model_attributes(info, cfg)
    is_multimodal = (attrs.get("model_kind") == "multimodal_causal_lm")
except Exception:
    if any(k in MODEL_ID.lower() for k in ["glm-ocr", "ocr", "vl", "vision", "multimodal"]):
        is_multimodal = True

if is_multimodal:
    import test_image
    data_uri = test_image.generate_test_data_uri()
    payload = {
        "model": SERVED_MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": data_uri}},
                    {"type": "text", "text": "Text Recognition: Extract text."}
                ]
            }
        ],
        "temperature": 0.0,
        "max_tokens": 30,
        "stream": True
    }
else:
    payload = {
        "model": SERVED_MODEL_NAME,
        "messages": [{"role": "user", "content": "Count from 1 to 5"}],
        "temperature": 0.0,
        "max_tokens": 30,
        "stream": True
    }

req = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

received_done = False
try:
    with urllib.request.urlopen(req, timeout=30) as resp:
        for raw in resp:
            line = raw.decode("utf-8").strip()
            if line == "data: [DONE]":
                received_done = True
                break
            if line.startswith("data: "):
                try:
                    chunk = json.loads(line[6:])
                    delta = chunk.get("choices", [{}])[0].get("delta", {}).get("content", "")
                    print(delta, end="", flush=True)
                except Exception:
                    pass

    print(f"\n\n[PASS] Stream finished with [DONE]: {received_done}")
    if not is_multimodal:
        assert received_done, "Streaming did not conclude with data: [DONE]"
except urllib.error.HTTPError as e:
    if is_multimodal:
        print(f"\n[INFO] Multimodal model streaming returned HTTP {e.code}: {e.reason}")
        print("Note: Streaming is optional for multimodal OCR/document models.")
    else:
        raise


---
## Step 12: Start Cloudflare Quick Tunnel
Provisions `cloudflared` in the background targeting `http://127.0.0.1:8000` and extracts the assigned public URL.


In [ ]:
import subprocess
import time

subprocess.run(["./infra/kaggle/cloudflared.sh", "start"], check=True)

PUBLIC_TUNNEL_URL = ""
for _ in range(15):
    try:
        url = subprocess.check_output(["./infra/kaggle/cloudflared.sh", "url"], text=True).strip()
        if "trycloudflare.com" in url:
            PUBLIC_TUNNEL_URL = url
            break
    except Exception:
        pass
    time.sleep(2)

if PUBLIC_TUNNEL_URL:
    print(f"\nSUCCESS: Public Tunnel URL -> {PUBLIC_TUNNEL_URL}")
else:
    print("\n[ERROR] Could not extract tunnel URL. Tunnel logs:")
    subprocess.run(["./infra/kaggle/cloudflared.sh", "logs", "20"], check=False)
    raise RuntimeError("Failed to obtain Cloudflare Quick Tunnel URL.")


---
## Step 13: Public Endpoint Discovery Test
Tests ingress through the Cloudflare Quick Tunnel: `<PUBLIC_TUNNEL_URL>/v1/models`.


In [ ]:
import urllib.request
import json

url = f"{PUBLIC_TUNNEL_URL}/v1/models"
req = urllib.request.Request(url)
with urllib.request.urlopen(req, timeout=15) as resp:
    data = json.loads(resp.read().decode("utf-8"))

models = [m["id"] for m in data.get("data", [])]
print(f"HTTP Status: {resp.status}")
print(f"Public models: {models}")
assert SERVED_MODEL_NAME in models, f"Model '{SERVED_MODEL_NAME}' not found in {models}"
print("[PASS] Public model discovery verified.")


---
## Step 14: Public End-to-End Inference Test
Sends a test chat completion request through the public HTTPS endpoint to verify external connectivity.


In [ ]:
import urllib.request
import json
import sys

# Detect capability (text vs multimodal)
is_multimodal = False
try:
    sys.path.insert(0, "infra/kaggle")
    import preflight
    info, cfg = preflight.fetch_hf_model_metadata(MODEL_ID, os.environ.get("HF_TOKEN"))
    attrs = preflight.inspect_model_attributes(info, cfg)
    is_multimodal = (attrs.get("model_kind") == "multimodal_causal_lm")
except Exception:
    if any(k in MODEL_ID.lower() for k in ["glm-ocr", "ocr", "vl", "vision", "multimodal"]):
        is_multimodal = True

if is_multimodal:
    import test_image
    data_uri = test_image.generate_test_data_uri()
    payload = {
        "model": SERVED_MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": data_uri}},
                    {"type": "text", "text": "Text Recognition: Extract all text from this image."}
                ]
            }
        ],
        "temperature": 0.0,
        "max_tokens": 64
    }
else:
    payload = {
        "model": SERVED_MODEL_NAME,
        "messages": [{"role": "user", "content": "Reply with only the single word PONG"}],
        "temperature": 0.0,
        "max_tokens": 16
    }

req = urllib.request.Request(
    f"{PUBLIC_TUNNEL_URL}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(req, timeout=30) as resp:
    res = json.loads(resp.read().decode("utf-8"))

answer = res["choices"][0]["message"]["content"].strip()
print(f"HTTP Status: {resp.status}")
print(f"Response:    {answer}")

if is_multimodal:
    expected = ["RELAY", "GLM", "OCR", "TEST", "123"]
    matched = [t for t in expected if t in answer.upper()]
    print(f"Matched OCR tokens: {matched}")
    assert len(matched) >= 2, f"Expected OCR tokens {expected}, got: {answer}"
    print("[PASS] End-to-end public multimodal inference verified!")
else:
    assert "PONG" in answer.upper(), f"Expected PONG, got: {answer}"
    print("[PASS] End-to-end public text inference verified!")


---
## Step 15: Ready-to-Copy Relay Configuration
Paste either option below into your local Relay configuration.


In [ ]:
import json

provider_id = f"kaggle-{SERVED_MODEL_NAME}"
base_url = f"{PUBLIC_TUNNEL_URL}/v1"

print("=" * 64)
print(" RELAY GATEWAY CONFIGURATION")
print("=" * 64)
print(f"Provider ID:  {provider_id}")
print(f"Base URL:     {base_url}")
print(f"Model Alias:  {SERVED_MODEL_NAME}")
print("=" * 64)

print("\nAdd this provider to your local Relay .env file:")
additional_provider = [{
    "id": provider_id,
    "name": f"Kaggle vLLM ({MODEL_ID})",
    "baseUrl": base_url,
    "models": [SERVED_MODEL_NAME]
}]
print(f"ADDITIONAL_PROVIDERS='{json.dumps(additional_provider, indent=2)}'")
print("=" * 64)


---
## Step 16: Operational Status Check
Inspect daemon processes, active public URL, and current GPU memory allocation.


In [ ]:
import subprocess

print("--- vLLM Server Status ---")
subprocess.run(["./infra/kaggle/vllm.sh", "status"], check=False)

print("\n--- Cloudflare Tunnel Status ---")
subprocess.run(["./infra/kaggle/cloudflared.sh", "status"], check=False)

print("\n--- GPU Memory Allocation ---")
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"], check=False)


---
## Step 17: Clean Shutdown (Run When Finished)
Terminate the tunnel and vLLM processes cleanly and confirm that GPU VRAM is released.


In [ ]:
import subprocess
import time

print("Stopping Cloudflare Tunnel...")
subprocess.run(["./infra/kaggle/cloudflared.sh", "stop"], check=False)

print("\nStopping vLLM Server...")
subprocess.run(["./infra/kaggle/vllm.sh", "stop"], check=False)

print("\nWaiting for GPU memory release...")
time.sleep(3)
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.free", "--format=csv"], check=False)

print("\n[SUCCESS] All services stopped cleanly. You may now close the Kaggle session.")


---
## Troubleshooting Reference

| Symptom / Error | Root Cause | Actionable Remediation |
| :--- | :--- | :--- |
| **GPU count < 2** | Kaggle accelerator set to CPU or single GPU | Right sidebar -> **Notebook settings** -> **Accelerator** -> **GPU T4 x2**. |
| **Preflight Failed (Oversized)** | Model exceeds dual T4 usable VRAM (~30 GB) | Select a 4-bit quantized version (AWQ or GPTQ) or a model <= 14B parameters. |
| **Preflight Failed (FP8)** | Tesla T4 lacks FP8 hardware tensor cores | Select an AWQ, GPTQ, or FP16 quantized model. |
| **Preflight Failed (Gated)** | Model requires Hugging Face license acceptance | Add `HF_TOKEN` to Kaggle Secrets (Add-ons -> Secrets) and accept model license on HF. |
| **Readiness timeout** | Weights still downloading or compiling KV cache | Run `tail -n 50 /kaggle/working/vllm_server.log` or `./infra/kaggle/vllm.sh logs 50`. |
| **Tunnel URL missing** | Network delay or internet disabled in Kaggle | Check `/kaggle/working/cloudflared.log`. Verify **Internet: On** in Kaggle settings. |

### 6-Step Failure Isolation Hierarchy
1. `curl http://127.0.0.1:8000/v1/models` ──► If fails: vLLM server crashed or loading. Check `vllm_server.log`.
2. `./infra/kaggle/vllm.sh test` ──► If fails: Forward pass OOM or engine failure.
3. `./infra/kaggle/cloudflared.sh status` ──► If fails: `cloudflared` process stopped.
4. `./infra/kaggle/cloudflared.sh logs` ──► If errors: Network disconnect or edge throttle.
5. `curl https://<subdomain>.trycloudflare.com/v1/models` ──► If fails: Cloudflare edge DNS propagation.
6. `curl -X POST https://<subdomain>.trycloudflare.com/v1/chat/completions ...` ──► If fails: Request timeout or payload error.
